In [ ]:
import osmnx as ox
import geopandas as gpd

# Pull Manhattan drivable network
G = ox.graph_from_place(
    "Manhattan, New York City, New York, USA",
    network_type="drive"
)

# Convert to GeoDataFrames
nodes, edges = ox.graph_to_gdfs(G)

# Inspect what you have
print(edges.columns.tolist())
print(edges['highway'].value_counts())
print(f"Total edges: {len(edges)}")

# Save to file
edges.to_file("data/raw/manhattan_roads.gpkg", driver="GPKG")
nodes.to_file("data/raw/manhattan_nodes.gpkg", driver="GPKG")

In [ ]:
from pathlib import Path
import urllib.request

import pandas as pd

# TLC parquet endpoints (same CDN linked from NYC Open Data / TLC trip record pages)
FHVHV_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-06.parquet"
ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

RAW_PATH = Path("../data/raw/fhvhv_tripdata_2025-06.parquet")
OUT_PATH = Path("../data/processed/fhvhv_tripdata_2025-06_manhattan_pu.parquet")
DESKTOP_RAW = Path(r"C:\Users\Arv\OneDrive\Desktop\Raw Taxi\fhvhv_tripdata_2025-06.parquet")


def download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {dest}")
    urllib.request.urlretrieve(url, dest)
    print("Download complete.")


def ensure_raw_parquet(path: Path) -> Path:
    if path.exists():
        print(f"Using cached raw file: {path}")
        return path
    if DESKTOP_RAW.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        print(f"Copying existing local file -> {path}")
        path.write_bytes(DESKTOP_RAW.read_bytes())
        return path
    download_file(FHVHV_URL, path)
    return path


ensure_raw_parquet(RAW_PATH)

# Official TLC taxi zone lookup: Borough == Manhattan -> PULocationID values
zones = pd.read_csv(ZONE_LOOKUP_URL)
manhattan_pu_ids = zones.loc[zones["Borough"] == "Manhattan", "LocationID"].astype("int32")
print(f"Manhattan pickup zones: {len(manhattan_pu_ids)} LocationIDs")

df = pd.read_parquet(RAW_PATH)
df_manhattan = df[df["PULocationID"].isin(manhattan_pu_ids)].copy()

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_manhattan.to_parquet(OUT_PATH, index=False)

print(f"All trips: {len(df):,}")
print(f"Manhattan pickups: {len(df_manhattan):,}")
print(f"Saved -> {OUT_PATH.resolve()}")
print(df_manhattan.dtypes)
df_manhattan.head()


## Yellow Taxi January 2010 (lat/lon coordinates)

Pre-2011 yellow trip records include raw pickup/dropoff longitude and latitude. Tries parquet first, then CSV if the parquet download fails.

In [ ]:
from pathlib import Path
import urllib.error
import urllib.request

import pandas as pd

YELLOW_PARQUET_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2010-01.parquet"
)
YELLOW_CSV_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2010-01.csv"
)

RAW_PARQUET = Path("../data/raw/yellow_tripdata_2010-01.parquet")
RAW_CSV = Path("../data/raw/yellow_tripdata_2010-01.csv")

COORD_COLS = [
    "pickup_longitude",
    "pickup_latitude",
    "dropoff_longitude",
    "dropoff_latitude",
]


def download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading {url} -> {dest}")
    urllib.request.urlretrieve(url, dest)
    print("Download complete.")


def load_yellow_2010() -> pd.DataFrame:
    if RAW_PARQUET.exists():
        print(f"Using cached parquet: {RAW_PARQUET}")
        return pd.read_parquet(RAW_PARQUET)

    try:
        download_file(YELLOW_PARQUET_URL, RAW_PARQUET)
        return pd.read_parquet(RAW_PARQUET)
    except (urllib.error.URLError, urllib.error.HTTPError, OSError) as err:
        print(f"Parquet download failed ({err}); trying CSV...")

    if RAW_CSV.exists():
        print(f"Using cached CSV: {RAW_CSV}")
    else:
        download_file(YELLOW_CSV_URL, RAW_CSV)

    return pd.read_csv(RAW_CSV, low_memory=False)


df_2010 = load_yellow_2010()

print("Column names:")
print(list(df_2010.columns))
print(f"\nShape: {df_2010.shape}")

present = [c for c in COORD_COLS if c in df_2010.columns]
missing = [c for c in COORD_COLS if c not in df_2010.columns]
print(f"\nCoordinate columns present: {present or '(none)'}")
print(f"Coordinate columns missing: {missing or '(none)'}")

if missing:
    print("\nReal non-zero coordinates: not available (columns missing).")
else:
    valid_pickup = (
        df_2010["pickup_longitude"].notna()
        & df_2010["pickup_latitude"].notna()
        & (df_2010["pickup_longitude"] != 0)
        & (df_2010["pickup_latitude"] != 0)
    )
    valid_dropoff = (
        df_2010["dropoff_longitude"].notna()
        & df_2010["dropoff_latitude"].notna()
        & (df_2010["dropoff_longitude"] != 0)
        & (df_2010["dropoff_latitude"] != 0)
    )

    print(
        f"\nReal non-zero pickup coordinates exist: {valid_pickup.any()} "
        f"({100 * valid_pickup.mean():.2f}% of rows)"
    )
    print(
        f"Real non-zero dropoff coordinates exist: {valid_dropoff.any()} "
        f"({100 * valid_dropoff.mean():.2f}% of rows)"
    )

    has_coords = valid_pickup & valid_dropoff
    print("\nSample (5 rows) — coordinate columns:")
    if has_coords.any():
        print(df_2010.loc[has_coords, COORD_COLS].head(5).to_string())
    else:
        print("(no rows with valid pickup and dropoff coordinates)")

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt

OUT_PATH_2010 = Path("../data/processed/tlc_2010_manhattan_pickups.parquet")

LON_MIN, LON_MAX = -74.0479, -73.9067
LAT_MIN, LAT_MAX = 40.6829, 40.8820

n_before = len(df_2010)

valid_pickup = (
    df_2010["pickup_longitude"].notna()
    & df_2010["pickup_latitude"].notna()
    & (df_2010["pickup_longitude"] != 0)
    & (df_2010["pickup_latitude"] != 0)
)

in_bbox = (
    df_2010["pickup_longitude"].between(LON_MIN, LON_MAX)
    & df_2010["pickup_latitude"].between(LAT_MIN, LAT_MAX)
)

df_mht_pu = df_2010.loc[valid_pickup & in_bbox].copy()
n_after = len(df_mht_pu)

print(f"Shape before filtering: {df_2010.shape}")
print(f"Shape after filtering:  {df_mht_pu.shape}")
print(f"% of original data retained: {100 * n_after / n_before:.2f}%")

print("\nPickup coordinate ranges after filtering:")
print(
    f"  pickup_longitude: {df_mht_pu['pickup_longitude'].min():.6f} "
    f"to {df_mht_pu['pickup_longitude'].max():.6f}"
)
print(
    f"  pickup_latitude:  {df_mht_pu['pickup_latitude'].min():.6f} "
    f"to {df_mht_pu['pickup_latitude'].max():.6f}"
)

gdf_mht_pu = gpd.GeoDataFrame(
    df_mht_pu,
    geometry=gpd.points_from_xy(
        df_mht_pu["pickup_longitude"],
        df_mht_pu["pickup_latitude"],
    ),
    crs="EPSG:4326",
)
gdf_mht_pu = gdf_mht_pu.to_crs("EPSG:2263")

OUT_PATH_2010.parent.mkdir(parents=True, exist_ok=True)
gdf_mht_pu.to_parquet(OUT_PATH_2010)
print(f"\nSaved -> {OUT_PATH_2010.resolve()}")

sample_n = min(10_000, len(df_mht_pu))
sample = df_mht_pu.sample(n=sample_n, random_state=42)

fig, ax = plt.subplots(figsize=(6, 8))
ax.scatter(
    sample["pickup_longitude"],
    sample["pickup_latitude"],
    s=0.5,
    alpha=0.4,
    c="steelblue",
)
ax.set_title(f"Manhattan pickups (random {sample_n:,} of {n_after:,})")
ax.set_xlabel("pickup_longitude")
ax.set_ylabel("pickup_latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## Clip 2010 pickups to Manhattan land boundary

Bounding-box filtering keeps points in the Hudson/East Rivers and outside the borough polygon. Clip to the OSM Manhattan place boundary (EPSG:2263).

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox

OUT_PATH_2010 = Path("../data/processed/tlc_2010_manhattan_pickups.parquet")

# Use in-memory GeoDataFrame if available; otherwise load saved file
if "gdf_mht_pu" not in dir():
    gdf_mht_pu = gpd.read_parquet(OUT_PATH_2010)

n_before = len(gdf_mht_pu)

# Manhattan borough boundary (OSM / Nominatim via osmnx)
manhattan_boundary = ox.geocode_to_gdf("Manhattan, New York, New York, USA")
manhattan_boundary = manhattan_boundary.to_crs(gdf_mht_pu.crs)

gdf_mht_pu_clean = gpd.sjoin(
    gdf_mht_pu,
    manhattan_boundary[["geometry"]],
    how="inner",
    predicate="within",
)
# Drop spatial-join index columns
gdf_mht_pu_clean = gdf_mht_pu_clean.drop(
    columns=[c for c in gdf_mht_pu_clean.columns if c.startswith("index_")],
    errors="ignore",
)

n_after = len(gdf_mht_pu_clean)
print(f"Shape before spatial join: {gdf_mht_pu.shape}")
print(f"Shape after spatial join:  {gdf_mht_pu_clean.shape}")
print(
    f"Rows removed: {n_before - n_after:,} "
    f"({100 * (n_before - n_after) / n_before:.2f}%)"
)

gdf_mht_pu = gdf_mht_pu_clean
gdf_mht_pu.to_parquet(OUT_PATH_2010)
print(f"\nOverwritten -> {OUT_PATH_2010.resolve()}")

sample_n = min(10_000, len(gdf_mht_pu))
sample = gdf_mht_pu.sample(n=sample_n, random_state=42)

fig, ax = plt.subplots(figsize=(6, 8))
manhattan_boundary.boundary.plot(ax=ax, color="black", linewidth=0.8, zorder=2)
sample.plot(ax=ax, markersize=0.5, alpha=0.4, color="steelblue", zorder=1)
ax.set_title(f"Clipped Manhattan pickups ({sample_n:,} of {n_after:,})")
ax.set_xlabel("Easting (ft, EPSG:2263)")
ax.set_ylabel("Northing (ft, EPSG:2263)")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()